# Chapter 2 – Simulated-Shift Coverage Forecasting: code

Reproduces the experiments of *Forecasting When Conformal Guarantees Break: Label-Free Coverage Alarms for Scientific Machine Learning Under Distribution Shift*.

**How to run (Colab):** Runtime → Run all. `QUICK = True` (default) uses 12 trials per model so the leave-one-family-out split (train seeds 0–9, test seeds ≥ 10) still works; set `QUICK = False` for the full 20 trials per model. Quick mode will not reproduce the exact numbers in the chapter.

In [ ]:
!pip -q install scikit-learn scipy joblib matplotlib pandas
QUICK = True
NSEEDS = 12 if QUICK else 20

## Experiments A and B: coverage under noise shift, weighted conformal, support-trust triage

In [ ]:
import numpy as np, pickle, warnings
from joblib import Parallel, delayed
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
X0, y0 = load_digits(return_X_y=True)
LEVELS = [0, 1, 2, 3, 4, 5]
ALPHAS = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]

def shift(X, s, rng):
    if s == 0: return X.copy()
    return np.clip((1 - 0.02 * s) * X + rng.normal(0, 0.35 * s, X.shape), 0, 16)

def wcp_sets(scores_cal, w_cal, w_test, P_test, alpha):
    """Weighted split conformal (Tibshirani et al. 2019), LAC score 1-p_y. Returns bool set matrix, vacuous mask."""
    order = np.argsort(scores_cal); sc = scores_cal[order]; cw = np.cumsum(w_cal[order]); S = w_cal.sum()
    n = len(w_test); K = P_test.shape[1]
    q = np.full(n, np.inf)
    for i in range(n):
        tot = S + w_test[i]
        idx = np.searchsorted(cw / tot, 1 - alpha, side="left")
        if idx < len(sc): q[i] = sc[idx]
    sets = (1 - P_test) <= q[:, None]
    return sets, np.isinf(q)

def trial(seed, model_name):
    rng = np.random.RandomState(seed)
    Xtr, Xr, ytr, yr = train_test_split(X0, y0, train_size=700, stratify=y0, random_state=seed)
    Xcal, Xte, ycal, yte = train_test_split(Xr, yr, train_size=450, stratify=yr, random_state=seed)
    mdl = (make_pipeline(StandardScaler(), MLPClassifier((64,), max_iter=300, random_state=seed)) if model_name == "MLP"
           else RandomForestClassifier(100, random_state=seed)).fit(Xtr, ytr)
    Pcal = mdl.predict_proba(Xcal); sc_cal = 1 - Pcal[np.arange(len(ycal)), ycal]
    out = []
    for s in LEVELS:
        Xs = shift(Xte, s, rng); P = mdl.predict_proba(Xs); pred = P.argmax(1); correct = pred == yte
        # cross-fitted domain classifier: 0=source(cal), 1=target(unlabeled test batch)
        Z = np.vstack([Xcal, Xs]); d = np.r_[np.zeros(len(Xcal)), np.ones(len(Xs))]
        dc = make_pipeline(StandardScaler(), LogisticRegression(C=0.003, max_iter=500))
        pd_ = cross_val_predict(dc, Z, d, cv=5, method="predict_proba")[:, 1]
        pd_ = np.clip(pd_, 1e-3, 1 - 1e-3)
        w_all = pd_ / (1 - pd_) * (len(Xcal) / len(Xs))
        w_cal, w_te = w_all[:len(Xcal)], w_all[len(Xcal):]
        trust_te = 1 - pd_[len(Xcal):]          # P(source | x): support trust
        ess = float(w_cal.sum() ** 2 / (w_cal ** 2).sum() / len(w_cal))
        rec = dict(seed=seed, model=model_name, s=s, acc=float(correct.mean()), ess=ess, auc=float(roc_auc_score(d, pd_)), conf=float(P.max(1).mean()))
        # sweep alpha for split CP and WCP
        for a in ALPHAS:
            for name, wc, wt in [("cp", np.ones(len(sc_cal)), np.ones(len(Xs))), ("wcp", w_cal, w_te)]:
                sets, vac = wcp_sets(sc_cal, wc, wt, P, a)
                cov = sets[np.arange(len(yte)), yte].mean(); size = sets.sum(1).mean()
                single = sets.sum(1) == 1
                rec[f"{name}_cov_{a}"] = float(cov); rec[f"{name}_size_{a}"] = float(size)
                rec[f"{name}_vac_{a}"] = float(vac.mean())
                rec[f"{name}_defer_{a}"] = float(1 - single.mean())
                rec[f"{name}_selacc_{a}"] = float(correct[single].mean()) if single.any() else np.nan
        # selective-accuracy curves for score-based triage
        pmax = P.max(1)
        for name, sc in [("msp", pmax), ("trust", pmax * trust_te), ("rand", rng.rand(len(pmax)))]:
            o = np.argsort(-sc); cs = np.cumsum(correct[o]) / np.arange(1, len(o) + 1)
            rec[f"curve_{name}"] = cs  # accuracy of the top-k accepted
        out.append(rec)
    return out

if __name__ == "__main__":
    jobs = [(sd, m) for m in ["MLP", "RandomForest"] for sd in range(NSEEDS)]
    res = Parallel(n_jobs=-1)(delayed(trial)(sd, m) for sd, m in jobs)
    pickle.dump([r for rr in res for r in rr], open("ch2.pkl", "wb")); print("ok")


In [ ]:
import pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
R = pd.DataFrame(pickle.load(open("ch2.pkl","rb")))
t = R.groupby(["model","s"])[["acc","conf","cp_cov_0.1","wcp_cov_0.1","ess"]].mean().round(3)
print(t)
fig, ax = plt.subplots(figsize=(6,3.4))
for m, g in R.groupby("model"):
    g = g.groupby("s")[["cp_cov_0.1","wcp_cov_0.1"]].mean()
    ax.plot(g.index, g["cp_cov_0.1"], marker="o", label=m+" split CP"); ax.plot(g.index, g["wcp_cov_0.1"], ls="--", marker="s", label=m+" weighted CP")
ax.axhline(0.9, color="k", lw=.8, ls=":"); ax.set_xlabel("shift severity"); ax.set_ylabel("coverage"); ax.legend(fontsize=7); plt.show()

## Experiment C: six shift families and the descriptors

In [ ]:
import numpy as np, pickle, warnings
from joblib import Parallel, delayed
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.neighbors import NearestNeighbors
from scipy.ndimage import uniform_filter
warnings.filterwarnings("ignore")
X0, y0 = load_digits(return_X_y=True)
FAMS = ["noise", "dropout", "gamma", "blur", "translation", "occlusion"]

def apply(fam, X, s, rng):
    if s == 0: return X.copy()
    n = len(X); I = X.reshape(n, 8, 8).copy()
    if fam == "noise": return np.clip(X + rng.normal(0, 0.35 * s, X.shape), 0, 16)
    if fam == "dropout": return X * (rng.rand(*X.shape) > 0.06 * s)
    if fam == "gamma": return 16 * (X / 16) ** (1 + 0.25 * s)
    if fam == "blur":
        b = min(0.15 * s, 0.75); B = np.stack([uniform_filter(i, 3, mode="constant") for i in I])
        return ((1 - b) * I + b * B).reshape(n, 64)
    if fam == "translation":
        k = (s + 1) // 2; out = np.empty_like(I)
        for j in range(n):
            dx, dy = rng.choice([-k, 0, k]), rng.choice([-k, 0, k]); out[j] = np.roll(I[j], (dx, dy), (0, 1))
        return out.reshape(n, 64)
    if fam == "occlusion":
        k = int(round(1 + 0.6 * s)); out = I.copy()
        for j in range(n):
            r, c = rng.randint(0, 8 - k + 1, 2); out[j, r:r + k, c:c + k] = 0
        return out.reshape(n, 64)

def ent(P): return float((-(P * np.log(P + 1e-12)).sum(1)).mean())

def trial(seed, model_name):
    rng = np.random.RandomState(seed)
    Xtr, Xr, ytr, yr = train_test_split(X0, y0, train_size=700, stratify=y0, random_state=seed)
    Xcal, Xte, ycal, yte = train_test_split(Xr, yr, train_size=450, stratify=yr, random_state=seed)
    mdl = (make_pipeline(StandardScaler(), MLPClassifier((64,), max_iter=300, random_state=seed)) if model_name == "MLP"
           else RandomForestClassifier(100, random_state=seed)).fit(Xtr, ytr)
    Pcal = mdl.predict_proba(Xcal); sc = np.sort(1 - Pcal[np.arange(len(ycal)), ycal])
    n = len(sc); q = sc[min(int(np.ceil((n + 1) * 0.9)) - 1, n - 1)]
    nn = NearestNeighbors(n_neighbors=1).fit(Xtr); dcal = nn.kneighbors(Xcal)[0].mean()
    conf0, ent0 = Pcal.max(1).mean(), ent(Pcal)
    rows = []
    for fam in FAMS:
        for s in range(0, 6):
            Xs = apply(fam, Xte, s, rng); P = mdl.predict_proba(Xs)
            cov = float(((1 - P[np.arange(len(yte)), yte]) <= q).mean())
            Z = np.vstack([Xcal, Xs]); d = np.r_[np.zeros(len(Xcal)), np.ones(len(Xs))]
            pdm = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(C=0.003, max_iter=300)), Z, d, cv=3, method="predict_proba")[:, 1]
            rows.append(dict(model=model_name, seed=seed, fam=fam, s=s, cov=cov, acc=float((P.argmax(1) == yte).mean()),
                             knn=float(nn.kneighbors(Xs)[0].mean() / dcal), dconf=float(conf0 - P.max(1).mean()),
                             dent=float(ent(P) - ent0), auc=float(roc_auc_score(d, pdm))))
    return rows

if __name__ == "__main__":
    jobs = [(sd, m) for m in ["MLP", "RandomForest"] for sd in range(NSEEDS)]
    res = Parallel(n_jobs=-1)(delayed(trial)(sd, m) for sd, m in jobs)
    pickle.dump([r for rr in res for r in rr], open("ch2b.pkl", "wb")); print("ok")


## Forecaster (leave-one-family-out) and alarm evaluation

In [ ]:
import pickle, numpy as np, json
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import roc_auc_score
r = pickle.load(open('ch2b.pkl','rb'))
FAMS = ["noise","dropout","gamma","blur","translation","occlusion"]
FS = {"full":["knn","dconf","dent","auc"],"knn":["knn"],"conf":["dconf"],"auc":["auc"]}
out = {}
rows=[]
for m in ["MLP","RandomForest"]:
    for f in FAMS:
        tr=[x for x in r if x['model']==m and x['fam']!=f and x['seed']<10]
        te=[x for x in r if x['model']==m and x['fam']==f and x['seed']>=10]
        yt=np.array([x['cov'] for x in te]); res={}
        res['nominal']=np.full(len(te),0.9)
        for k,cols in FS.items():
            A=np.array([[x[c] for c in cols] for x in tr]); B=np.array([[x[c] for c in cols] for x in te])
            g=GradientBoostingRegressor(n_estimators=150,max_depth=2,learning_rate=0.05,subsample=0.8,random_state=0).fit(A,[x['cov'] for x in tr])
            res[k]=g.predict(B)
        for k,p in res.items():
            rows.append(dict(model=m,fam=f,method=k,mae=float(np.abs(p-yt).mean()),
              auc=float(roc_auc_score(yt<0.8,-p)) if k!='nominal' and 0<(yt<0.8).sum()<len(yt) else None,
              true=yt.tolist(),pred=p.tolist(),s=[x['s'] for x in te]))
json.dump(rows,open('ch2b.json','w'))
for m in ["MLP","RandomForest"]:
    print(m)
    for k in ["nominal","auc","conf","knn","full"]:
        R=[x for x in rows if x['model']==m and x['method']==k]
        print(f"  {k:8s} MAE {np.mean([x['mae'] for x in R]):.3f}  per-fam "+" ".join(f"{x['mae']:.3f}" for x in R)+"  AUC "+(" ".join(f"{x['auc']:.2f}" if x['auc'] is not None else 'na' for x in R)))
# raw descriptors vs severity
for m in ["MLP","RandomForest"]:
    R=[x for x in r if x['model']==m and x['fam']=='noise']
    for s in [0,2,5]:
        Q=[x for x in R if x['s']==s]; print(m,s,*(f"{k}={np.mean([x[k] for x in Q]):.3f}" for k in ['cov','acc','knn','dconf','dent','auc']))


In [ ]:
import json, numpy as np
rows = json.load(open("ch2b.json"))
print("Mean absolute error of the coverage forecast (mean over held-out families)")
for m in ["MLP","RandomForest"]:
    print(m, {k: round(float(np.mean([x["mae"] for x in rows if x["model"]==m and x["method"]==k])),3) for k in ["nominal","auc","conf","knn","full"]})
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, m in zip(axs, ["MLP","RandomForest"]):
    t = np.concatenate([x["true"] for x in rows if x["model"]==m and x["method"]=="full"]); p = np.concatenate([x["pred"] for x in rows if x["model"]==m and x["method"]=="full"])
    ax.scatter(t, p, s=8, alpha=.5); ax.plot([0,1],[0,1],"k-",lw=.8); ax.axhline(.8, ls=":", c="k"); ax.axvline(.8, ls=":", c="k"); ax.set_title(m); ax.set_xlabel("true coverage")
axs[0].set_ylabel("forecast coverage"); plt.show()